# Latent-dynamics model for battery degradation curves
**Goal:** from the early reference-performance-test (RPT) curves Q(V) of a cell plus its usage
conditions, predict the *whole Q(V) curve* at any later week by integrating a learned ODE.

```
 early curves + conditions --encoder--> z(t0) --- dz/dτ = f(z, conditions) --RK4--> z(t) --decoder--> ΔQ(V, t)
```
* `Q_pred(t) = Q(t0) + D(z(t)) − D(z(t0))` -> exact at the anchor, the decoder only has to describe *change*.
* The ODE right-hand side sees the conditions (Cchg, Cdchg, DoD, ...) so usage sets the **rate** of ageing.
* **Training uses full rollout loss** over a horizon that grows during training (curriculum), not one-step loss.
* **Sliding origin:** every RPT in `ORIGIN_WEEKS` is used as a starting point, which multiplies the data.
* A small **deep ensemble** gives uncertainty bands.
* Evaluated from origin = week 3 on held-out cells, against two baselines (persistence, linear extrapolation).

Needs PyTorch. Inputs: `Q_interpolated/*.csv` (1000-point Q(V) per RPT) and `Valid_cells.csv`.

In [ ]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------------- settings ----------------
DATA_DIR = "PATH/TO/ISU-ILCC_dataset"                    # <-- EDIT
Q_DIR = os.path.join(DATA_DIR, "Q_interpolated")
VALID_CELLS_CSV = os.path.join(DATA_DIR, "Valid_cells.csv")
OUT_DIR = "latent_ode_outputs"

Q_TO_MAH = 1000.0           # Q_interpolated is in Ah
V_STRIDE = 10               # 1000 -> 100 voltage points
EVAL_ORIGIN = 3             # the "early" window ends at this week for evaluation
ORIGIN_WEEKS = list(range(3, 13))   # training origins (sliding origin); use [3] for strictly-early-only
HMAX = 40                   # max rollout horizon in weeks
MIN_RET = 0.65              # ignore RPTs after capacity retention falls below this
MIN_FUTURE = 3              # a sample needs at least this many future RPTs

LATENT = 8
HIDDEN = 128
TIME_SCALE = 10.0           # weeks per unit of ODE time (keeps the ODE well scaled)
SUBSTEPS = 1                # RK4 steps per week
EPOCHS = 150
BATCH = 128
LR = 2e-3
WEIGHT_DECAY = 1e-4
H0 = 4                      # curriculum: start horizon (weeks) ...
CURR_FRAC = 0.7             # ... grown to HMAX over this fraction of training
LAMBDA_CAP = 1.0            # extra weight on capacity (= max of Q) error
ENSEMBLE = 3
SPLIT_BY_GROUP = False      # True = hold out whole cycling groups (harder, honest)
TEST_FRAC, VAL_FRAC = 0.2, 0.2
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float32)
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE)

# group -> (Cchg, Cdchg, DoD %)   (Table A1 of the dataset README)
CONDITIONS = {
    1: (0.5, 0.5, 25.90), 2: (0.5, 0.5, 46.11), 3: (0.5, 0.5, 67.77),
    4: (1.0, 0.5, 47.27), 5: (1.0, 0.5, 97.54), 6: (2.0, 0.5, 27.23),
    7: (2.0, 0.5, 46.86), 8: (2.0, 0.5, 67.24), 9: (2.0, 0.5, 96.93),
    10: (2.5, 0.5, 47.21), 11: (2.5, 0.5, 95.98), 12: (3.0, 0.5, 28.60),
    13: (3.0, 0.5, 47.26), 14: (3.0, 0.5, 66.32), 16: (0.5, 0.5, 97.62),
    17: (1.0, 0.5, 67.53), 18: (2.5, 0.5, 27.53), 19: (2.5, 0.5, 65.70),
    20: (0.8, 0.5, 98.16), 21: (1.2, 0.5, 97.98), 22: (1.4, 0.5, 97.88),
    23: (1.6, 0.5, 97.61), 24: (1.8, 0.5, 97.51), 25: (1.8, 0.6, 51.43),
    26: (1.4, 2.2, 4.16), 27: (0.6, 2.4, 57.85), 28: (2.4, 1.6, 80.99),
    29: (1.6, 1.8, 52.48), 30: (0.8, 0.8, 77.59), 31: (1.2, 1.0, 61.43),
    32: (1.0, 1.4, 38.86), 33: (2.0, 1.2, 40.65), 34: (2.2, 2.0, 27.60),
    35: (1.825, 0.5, 97.02), 36: (2.075, 0.5, 97.06), 37: (0.725, 0.5, 98.08),
    38: (1.875, 0.5, 95.83), 39: (1.475, 0.5, 97.14), 40: (1.825, 1.025, 31.94),
    41: (2.075, 1.775, 67.02), 42: (0.725, 2.375, 36.32), 43: (1.875, 2.325, 38.00),
    44: (0.775, 1.275, 37.12), 45: (1.125, 1.725, 80.26), 46: (1.225, 2.025, 75.36),
    47: (2.325, 1.925, 48.27), 48: (2.375, 2.225, 58.57), 49: (0.975, 0.675, 18.82),
    50: (2.425, 1.625, 20.10), 51: (2.275, 1.875, 53.64), 52: (1.425, 0.875, 83.55),
    53: (2.025, 0.825, 91.54), 54: (0.925, 1.125, 27.05), 55: (1.025, 2.475, 61.44),
    56: (2.175, 0.975, 53.81), 57: (1.775, 1.175, 17.42), 58: (2.475, 0.575, 42.03),
    59: (1.325, 1.825, 31.64), 60: (0.675, 1.325, 83.63), 61: (2.125, 1.975, 54.07),
    62: (1.575, 2.425, 64.19), 63: (1.975, 1.675, 80.57), 64: (1.175, 1.425, 91.44),
}

## 1. Load cells: RPT times, downsampled Q(V), conditions

In [ ]:
V_FULL = np.linspace(3.0, 4.2, 1000)
V_DS = V_FULL[::V_STRIDE]
NV = len(V_DS)

def group_of(cell):
    return int(re.match(r"G(\d+)C", cell).group(1))

def rpt_weeks(group, n):
    return np.array(([0] + ([0.5] if group >= 20 else []) + list(range(1, 80)))[:n], dtype=float)

def cond_vector(group):
    cchg, cdchg, dod = CONDITIONS[group][0], CONDITIONS[group][1], CONDITIONS[group][2] / 100.0
    cpw = 168.0 / (dod / cchg + dod / cdchg)                 # ideal cycles per week
    return np.array([cchg, cdchg, dod, np.sqrt(cchg * dod), np.sqrt(cdchg * dod), cpw, cpw * dod], dtype=np.float64)
COND_NAMES = ["Cchg", "Cdchg", "DoD", "sqrt(Cchg*DoD)", "sqrt(Cdchg*DoD)", "cycles/wk", "EFC/wk"]

def load_cell(cell):
    d = pd.read_csv(os.path.join(Q_DIR, f"{cell}.csv"), header=None).apply(pd.to_numeric, errors="coerce")
    d = d.dropna(axis=1, how="any")
    Q = d.values.T * Q_TO_MAH                              # (n_rpt, 1000)
    g = group_of(cell)
    w = rpt_weeks(g, Q.shape[0])
    Qds = Q[:, ::V_STRIDE]
    cap = Qds.max(axis=1)
    ret = cap / cap[0]
    below = np.where(ret < MIN_RET)[0]
    keep = len(w) if len(below) == 0 else below[0]         # drop everything once retention < MIN_RET
    keep = max(keep, 1)
    return dict(cell=cell, group=g, weeks=w[:keep], Q=Qds[:keep], cap=cap[:keep], cond=cond_vector(g))

cells = pd.read_csv(VALID_CELLS_CSV, header=None).values.ravel()
cells = sorted({str(c).strip() for c in cells if re.match(r"^G\d+C\d+$", str(c).strip())},
               key=lambda c: (group_of(c), c))
data = {}
for c in cells:
    try:
        r = load_cell(c)
        if (r["weeks"] >= EVAL_ORIGIN + MIN_FUTURE).any():
            data[c] = r
    except Exception as e:
        print(f"skip {c}: {type(e).__name__}: {e}")
cell_ids = np.array(sorted(data, key=lambda c: (group_of(c), c)))
groups = np.array([data[c]["group"] for c in cell_ids])
print(f"{len(cell_ids)} usable cells; RPTs per cell: median {np.median([len(data[c]['weeks']) for c in cell_ids]):.0f}")

## 2. Split by cell

In [ ]:
rng = np.random.default_rng(SEED)
if SPLIT_BY_GROUP:
    ug = rng.permutation(np.unique(groups))
    n_te, n_va = int(round(TEST_FRAC * len(ug))), int(round(VAL_FRAC * len(ug)))
    g_te, g_va, g_tr = ug[:n_te], ug[n_te:n_te + n_va], ug[n_te + n_va:]
    split = {"train": cell_ids[np.isin(groups, g_tr)], "val": cell_ids[np.isin(groups, g_va)],
             "test": cell_ids[np.isin(groups, g_te)]}
else:
    p = rng.permutation(len(cell_ids))
    n_te, n_va = int(round(TEST_FRAC * len(p))), int(round(VAL_FRAC * len(p)))
    split = {"test": cell_ids[p[:n_te]], "val": cell_ids[p[n_te:n_te + n_va]], "train": cell_ids[p[n_te + n_va:]]}
print({k: len(v) for k, v in split.items()})

## 3. Samples: (cell, origin RPT) -> future curves at +1, +2, ... weeks

In [ ]:
def make_samples(ids, origins):
    rows = dict(q_m=[], dq0=[], w=[], cond=[], Y=[], mask=[], cap=[], cell=[], origin=[])
    for c in ids:
        r = data[c]
        wk = r["weeks"]
        for w0 in origins:
            hit = np.where(wk == w0)[0]
            if len(hit) == 0:
                continue
            m = hit[0]
            fut = [j for j in range(m + 1, len(wk))
                   if float(wk[j]).is_integer() and 0 < wk[j] - w0 <= HMAX]
            if len(fut) < MIN_FUTURE:
                continue
            Y = np.zeros((HMAX, NV)); mask = np.zeros(HMAX); cap = np.zeros(HMAX)
            for j in fut:
                h = int(wk[j] - w0) - 1
                Y[h] = r["Q"][j] - r["Q"][m]; mask[h] = 1; cap[h] = r["cap"][j]
            rows["q_m"].append(r["Q"][m]); rows["dq0"].append(r["Q"][m] - r["Q"][0])
            rows["w"].append(w0); rows["cond"].append(r["cond"])
            rows["Y"].append(Y); rows["mask"].append(mask); rows["cap"].append(cap)
            rows["cell"].append(c); rows["origin"].append(w0)
    return {k: (np.array(v) if k not in ("cell",) else np.array(v)) for k, v in rows.items()}

S = {"train": make_samples(split["train"], ORIGIN_WEEKS),
     "val": make_samples(split["val"], [EVAL_ORIGIN]),
     "test": make_samples(split["test"], [EVAL_ORIGIN])}
print({k: len(v["w"]) for k, v in S.items()}, "samples")

# standardisation from TRAIN samples only
tr = S["train"]
enc_raw = lambda s: np.concatenate([s["q_m"], s["dq0"], s["w"][:, None]], axis=1)
E_mu, E_sd = enc_raw(tr).mean(0), enc_raw(tr).std(0) + 1e-6
C_mu, C_sd = tr["cond"].mean(0), tr["cond"].std(0) + 1e-6
Y_sd = float(np.sqrt((tr["Y"] ** 2).sum() / (tr["mask"].sum() * NV)))     # scale of dQ in mAh
print(f"typical |dQ| scale: {Y_sd:.2f} mAh")

def to_tensors(s):
    X = torch.tensor((enc_raw(s) - E_mu) / E_sd, dtype=torch.float32, device=DEVICE)
    C = torch.tensor((s["cond"] - C_mu) / C_sd, dtype=torch.float32, device=DEVICE)
    T = dict(X=X, C=C,
             q_m=torch.tensor(s["q_m"], dtype=torch.float32, device=DEVICE),
             Y=torch.tensor(s["Y"] / Y_sd, dtype=torch.float32, device=DEVICE),
             mask=torch.tensor(s["mask"], dtype=torch.float32, device=DEVICE),
             cap=torch.tensor(s["cap"], dtype=torch.float32, device=DEVICE))
    return T
T = {k: to_tensors(v) for k, v in S.items()}

## 4. The model: encoder -> neural ODE (RK4) -> decoder

In [ ]:
def mlp(i, h, o, layers=2, act=nn.SiLU):
    mods, d = [], i
    for _ in range(layers):
        mods += [nn.Linear(d, h), act()]; d = h
    mods.append(nn.Linear(d, o))
    return nn.Sequential(*mods)

class LatentODE(nn.Module):
    def __init__(self, n_enc, n_cond, nv, latent=LATENT, hidden=HIDDEN):
        super().__init__()
        self.enc = mlp(n_enc, hidden, latent)
        self.f = nn.Sequential(nn.Linear(latent + n_cond, hidden), nn.Tanh(),
                               nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, latent))
        self.dec = mlp(latent, hidden, nv)
        with torch.no_grad():                      # start with slow, stable dynamics
            self.f[-1].weight.mul_(0.1); self.f[-1].bias.zero_()
        self.dt = 1.0 / SUBSTEPS / TIME_SCALE

    def _rk4(self, z, c):
        f, dt = self.f, self.dt
        fz = lambda zz: f(torch.cat([zz, c], -1))
        k1 = fz(z); k2 = fz(z + 0.5 * dt * k1); k3 = fz(z + 0.5 * dt * k2); k4 = fz(z + dt * k3)
        return z + dt / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)

    def forward(self, X, C, H, C_dyn=None):
        """Returns dQ (B, H, NV) in normalised units and the latent trajectory (B, H, latent)."""
        c = C if C_dyn is None else C_dyn
        z = self.enc(X); z0 = z; zs = []
        for _ in range(H):
            for _ in range(SUBSTEPS):
                z = self._rk4(z, c)
            zs.append(z)
        Z = torch.stack(zs, 1)
        dq = self.dec(Z) - self.dec(z0).unsqueeze(1)           # exact zero change at the anchor
        return dq, Z

def loss_fn(model, B, H):
    dq, _ = model(B["X"], B["C"], H)
    m = B["mask"][:, :H]
    mse = (m.unsqueeze(-1) * (dq - B["Y"][:, :H]) ** 2).sum() / (m.sum() * NV + 1e-9)
    cap_pred = (B["q_m"].unsqueeze(1) + dq * Y_sd).max(-1).values          # mAh
    cap_err = ((cap_pred - B["cap"][:, :H]) / Y_sd) ** 2
    cap_loss = (m * cap_err).sum() / (m.sum() + 1e-9)
    return mse + LAMBDA_CAP * cap_loss, mse.detach(), cap_loss.detach()

## 5. Training with rollout loss and a horizon curriculum

In [ ]:
def train_one(seed, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LatentODE(T["train"]["X"].shape[1], T["train"]["C"].shape[1], NV).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * int(np.ceil(len(T["train"]["X"]) / BATCH)))
    N = len(T["train"]["X"]); hist = []; best = (np.inf, None)
    for ep in range(EPOCHS):
        frac = min(1.0, ep / max(1, CURR_FRAC * EPOCHS))
        H = int(round(H0 + (HMAX - H0) * frac))
        model.train(); perm = torch.randperm(N, device=DEVICE); tot = 0.0
        for i in range(0, N, BATCH):
            idx = perm[i:i + BATCH]
            B = {k: v[idx] for k, v in T["train"].items()}
            loss, _, _ = loss_fn(model, B, H)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model, T["val"], HMAX)[0].item() if len(T["val"]["X"]) else np.nan
        hist.append((ep, H, tot / N, vl))
        if H == HMAX and vl < best[0]:
            best = (vl, {k: v.clone() for k, v in model.state_dict().items()})
        if verbose and (ep % 10 == 0 or ep == EPOCHS - 1):
            print(f"seed {seed} ep {ep:3d}  H={H:2d}  train {tot / N:.4f}  val(H={HMAX}) {vl:.4f}")
    if best[1] is not None:
        model.load_state_dict(best[1])
    return model, pd.DataFrame(hist, columns=["epoch", "horizon", "train_loss", "val_loss"])

t0 = time.time()
models, hists = [], []
for k in range(ENSEMBLE):
    m, h = train_one(SEED + k)
    models.append(m); hists.append(h)
print(f"trained {ENSEMBLE} models in {(time.time() - t0) / 60:.1f} min")
torch.save([m.state_dict() for m in models], os.path.join(OUT_DIR, "latent_ode_ensemble.pt"))

## 6. Evaluation from origin = week 3 on held-out cells
Curve error is RMSE over the voltage window (mAh); capacity = max of the predicted Q(V).

In [ ]:
@torch.no_grad()
def predict(split_name, C_dyn=None):
    outs, Zs = [], []
    for m in models:
        m.eval(); dq, Z = m(T[split_name]["X"], T[split_name]["C"], HMAX, C_dyn); outs.append(dq.cpu().numpy() * Y_sd); Zs.append(Z.cpu().numpy())
    outs = np.stack(outs)                                   # (ens, N, H, NV) in mAh
    return outs.mean(0), outs.std(0), np.mean(Zs, 0)

def curves_and_caps(s, dq):
    Q = s["q_m"][:, None, :] + dq
    return Q, Q.max(-1)

def score(s, dq, label):
    Q, cap = curves_and_caps(s, dq)
    Qt = s["q_m"][:, None, :] + s["Y"]
    m = s["mask"].astype(bool)
    curve_rmse = np.sqrt(((Q - Qt) ** 2)[m].mean())
    err = (cap - s["cap"])[m]
    mape = np.mean(np.abs(err) / s["cap"][m]) * 100
    ss = ((s["cap"][m] - s["cap"][m].mean()) ** 2).sum()
    return dict(model=label, curve_RMSE_mAh=curve_rmse, cap_RMSE_mAh=np.sqrt((err ** 2).mean()),
                cap_MAPE_pct=mape, cap_R2=1 - (err ** 2).sum() / ss)

s_te = S["test"]
dq_mean, dq_std, Z_te = predict("test")
h_axis = np.arange(1, HMAX + 1)
dq_persist = np.zeros_like(dq_mean)                                     # curve stays as at origin
dq_lin = s_te["dq0"][:, None, :] * (h_axis[None, :, None] / s_te["w"][:, None, None])   # keep early drift rate
res = pd.DataFrame([score(s_te, dq_mean, "latent ODE (ensemble mean)"),
                    score(s_te, dq_persist, "persistence (no change)"),
                    score(s_te, dq_lin, "linear extrapolation of early drift")]).set_index("model")
res.to_csv(os.path.join(OUT_DIR, "test_metrics.csv"))
print(res.round(3).to_string())

## 7. Plots

In [ ]:
# 7a. training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for h in hists:
    ax[0].plot(h.epoch, h.train_loss, alpha=0.7); ax[1].plot(h.epoch, h.val_loss, alpha=0.7)
ax[0].set_yscale("log"); ax[1].set_yscale("log")
ax[0].set_title("Train loss (horizon grows by curriculum)"); ax[1].set_title(f"Validation loss at full horizon H={HMAX}")
for a in ax: a.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "training.png"), dpi=150); plt.show()

# 7b. capacity vs week for test cells, with ensemble band
Qp, capp = curves_and_caps(s_te, dq_mean)
_, cap_lo = curves_and_caps(s_te, dq_mean - 2 * dq_std)
_, cap_hi = curves_and_caps(s_te, dq_mean + 2 * dq_std)
show = np.arange(min(8, len(s_te["w"])))
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
for ax, i in zip(axes.ravel(), show):
    c = s_te["cell"][i]; r = data[c]; mk = s_te["mask"][i].astype(bool)
    ax.plot(r["weeks"], r["cap"], "ko-", ms=3, lw=1, label="measured")
    wk = EVAL_ORIGIN + h_axis
    ax.plot(wk, capp[i], "r-", lw=2, label="latent ODE")
    ax.fill_between(wk, cap_lo[i], cap_hi[i], color="r", alpha=0.2)
    ax.plot(wk, (s_te["q_m"][i][None, :] + dq_lin[i]).max(-1), "b--", lw=1, label="linear extrap.")
    ax.axvspan(0, EVAL_ORIGIN, color="grey", alpha=0.2)
    ax.set_xlim(0, max(r["weeks"].max(), wk[mk].max()) * 1.05); ax.set_title(c, fontsize=9); ax.set_xlabel("weeks")
axes[0, 0].legend(fontsize=7); axes[0, 0].set_ylabel("capacity [mAh]"); axes[1, 0].set_ylabel("capacity [mAh]")
fig.suptitle("Capacity rollout from week 3 (grey = observed window); band = ±2σ across the ensemble", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "capacity_rollouts.png"), dpi=150); plt.show()

# 7c. the actual curves: Q(V) at several future weeks for one test cell
i = 0; c = s_te["cell"][i]; mk = np.where(s_te["mask"][i] > 0)[0]
picks = [mk[len(mk) // 4], mk[len(mk) // 2], mk[-1]] if len(mk) >= 3 else list(mk)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(V_DS, s_te["q_m"][i], "k", lw=1.5, label=f"week {EVAL_ORIGIN} (input)")
for j, col in zip(picks, ["tab:blue", "tab:green", "tab:red"]):
    ax[0].plot(V_DS, s_te["q_m"][i] + s_te["Y"][i, j], c=col, lw=1.5, label=f"measured +{j + 1}w")
    ax[0].plot(V_DS, Qp[i, j], c=col, ls="--", lw=1.5)
    ax[1].plot(V_DS, Qp[i, j] - (s_te["q_m"][i] + s_te["Y"][i, j]), c=col, label=f"+{j + 1}w")
ax[0].set_xlabel("V"); ax[0].set_ylabel("Q [mAh]"); ax[0].set_title(f"{c}: solid = measured, dashed = predicted"); ax[0].legend(fontsize=7)
ax[1].axhline(0, c="k", lw=0.8); ax[1].set_xlabel("V"); ax[1].set_ylabel("Predicted - measured [mAh]"); ax[1].set_title("Curve error"); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "curve_prediction.png"), dpi=150); plt.show()

# 7d. error vs horizon
def per_h(s, dq):
    Q, cap = curves_and_caps(s, dq); Qt = s["q_m"][:, None, :] + s["Y"]; m = s["mask"]
    cr = np.sqrt((((Q - Qt) ** 2).mean(-1) * m).sum(0) / np.maximum(m.sum(0), 1))
    cm = ((np.abs(cap - s["cap"]) / np.maximum(s["cap"], 1)) * m).sum(0) / np.maximum(m.sum(0), 1) * 100
    ok = m.sum(0) >= 3
    return np.where(ok, cr, np.nan), np.where(ok, cm, np.nan)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for dq, lab, st in ((dq_mean, "latent ODE", "r-"), (dq_persist, "persistence", "g:"), (dq_lin, "linear extrap.", "b--")):
    cr, cm = per_h(s_te, dq); ax[0].plot(h_axis, cr, st, label=lab); ax[1].plot(h_axis, cm, st, label=lab)
ax[0].set_ylabel("Curve RMSE [mAh]"); ax[1].set_ylabel("Capacity MAPE [%]")
for a in ax: a.set_xlabel("Rollout horizon [weeks beyond week 3]")
ax[0].legend(); fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "error_vs_horizon.png"), dpi=150); plt.show()

# 7e. capacity parity
m = s_te["mask"].astype(bool)
fig, ax = plt.subplots(figsize=(5, 5))
sc = ax.scatter(s_te["cap"][m], capp[m], c=np.broadcast_to(h_axis, m.shape)[m], s=8, cmap="viridis")
lo, hi = s_te["cap"][m].min() * 0.97, s_te["cap"][m].max() * 1.03
ax.plot([lo, hi], [lo, hi], "k-"); ax.set_xlabel("Measured capacity [mAh]"); ax.set_ylabel("Predicted [mAh]")
fig.colorbar(sc, label="horizon [weeks]"); fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "capacity_parity.png"), dpi=150); plt.show()

# 7f. latent trajectories (PCA of the ensemble-mean latent state), coloured by DoD and Cchg
Zf = Z_te.reshape(-1, Z_te.shape[-1]); Zc = Zf - Zf.mean(0)
U, Sg, Vt = np.linalg.svd(Zc, full_matrices=False); P = (Zc @ Vt[:2].T).reshape(Z_te.shape[0], HMAX, 2)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
for a, k, name in ((ax[0], 2, "DoD"), (ax[1], 0, "Cchg")):
    cv = s_te["cond"][:, k]
    for i in range(len(P)):
        ok = s_te["mask"][i] > 0
        a.plot(P[i, ok, 0], P[i, ok, 1], "-", c=plt.cm.viridis((cv[i] - cv.min()) / (np.ptp(cv) + 1e-9)), lw=1)
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(cv.min(), cv.max())); fig.colorbar(sm, ax=a, label=name)
    a.set_xlabel("latent PC1"); a.set_ylabel("latent PC2"); a.set_title(f"Latent trajectories coloured by {name}")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "latent_trajectories.png"), dpi=150); plt.show()

## 8. What-if: same cell, different usage
Keep the encoder input (what the cell looks like at week 3) fixed and swap only the conditions that drive the ODE.
Caution: this is only meaningful to the extent the model learned condition-dependence from *different cells*;
treat it as a hypothesis generator, not a measurement.

In [ ]:
def what_if(i, changes, label):
    c = s_te["cond"][i].copy()
    for k, v in changes.items():
        c[COND_NAMES.index(k)] = v
    cchg, cdchg, dod = c[0], c[1], c[2]
    c[3], c[4] = np.sqrt(cchg * dod), np.sqrt(cdchg * dod)
    c[5] = 168.0 / (dod / cchg + dod / cdchg); c[6] = c[5] * dod
    Cd = torch.tensor((c - C_mu) / C_sd, dtype=torch.float32, device=DEVICE)[None].repeat(len(T["test"]["X"]), 1)
    dq, _, _ = predict("test", C_dyn=Cd)
    return curves_and_caps(s_te, dq)[1][i], label

i = 0
base_c = s_te["cond"][i]
scen = [what_if(i, {}, f"as tested (Cchg {base_c[0]:.2f}, Cdchg {base_c[1]:.2f}, DoD {base_c[2]:.2f})"),
        what_if(i, {"Cchg": max(0.5, base_c[0] / 2)}, "half the charge rate"),
        what_if(i, {"Cchg": min(3.0, base_c[0] * 2)}, "double the charge rate"),
        what_if(i, {"DoD": max(0.05, base_c[2] * 0.5)}, "half the depth of discharge")]
fig, ax = plt.subplots(figsize=(7, 4.3))
cap0 = data[s_te["cell"][i]]["cap"][0]
for cap_i, lab in scen:
    ok = np.cumprod(cap_i >= MIN_RET * cap0).astype(bool)      # stop where the model was never trained (retention < MIN_RET)
    ax.plot((EVAL_ORIGIN + h_axis)[ok], cap_i[ok], lw=2, label=lab)
ax.plot(data[s_te["cell"][i]]["weeks"], data[s_te["cell"][i]]["cap"], "ko", ms=3, label="measured (as tested)")
ax.set_xlabel("weeks"); ax.set_ylabel("capacity [mAh]"); ax.legend(fontsize=7)
ax.set_title(f"What-if for {s_te['cell'][i]}")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "what_if.png"), dpi=150); plt.show()